# 2. Document와 문서 분할

**시나리오:** 긴 HR 규정을 검색 가능한 작은 근거 단위로 나눕니다.

**학습 목표:** LangChain `Document`, metadata, `RecursiveCharacterTextSplitter`와 `chunk_id`의 역할을 익힙니다.

## 중요 변수·함수

- `POLICY_SOURCE_DOCUMENTS`: 분할 전 원문과 `document_id`를 보존합니다.
- `split_policy_documents()`: overlap을 둔 chunk를 만들고 안정적인 인용 ID를 붙입니다.
- `chunk_size`: 검색 근거의 크기와 문맥 손실 사이를 조절합니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# canonical 함수로 같은 원문을 서로 다른 크기로 분할합니다.
from week1.app import POLICY_SOURCE_DOCUMENTS, split_policy_documents

small_chunks = split_policy_documents(POLICY_SOURCE_DOCUMENTS, chunk_size=70)
large_chunks = split_policy_documents(POLICY_SOURCE_DOCUMENTS, chunk_size=160)
(len(POLICY_SOURCE_DOCUMENTS), len(small_chunks), len(large_chunks))

In [ ]:
# 인용 가능성을 위해 metadata가 유지·확장됐는지 확인합니다.
first_chunk = small_chunks[0]
assert first_chunk.metadata['document_id'] == 'hr-leave-policy'
assert first_chunk.metadata['chunk_id'].startswith('hr-leave-policy-')
{'text': first_chunk.page_content, 'metadata': first_chunk.metadata}

## 예측 과제와 해석

**예측 과제:** `chunk_size`를 줄이면 chunk 수와 검색 정밀도가 어떻게 변할지 예상하세요.

**해석:** 작은 chunk는 정밀한 인용에 유리하지만 문맥이 잘릴 수 있습니다. 따라서 크기는 고정 정답이 아니라 문서와 평가 결과로 결정합니다.